# Lecture 0 · Live demo — why a naive pipeline dies at 64 clients

**LLM Serving Mastery** · runtime: **Google Colab, GPU → T4** (Runtime → Change runtime type → T4 GPU).

Same model, same GPU, two servers, one load generator:

1. **Naive server** — `transformers.pipeline` behind a tiny OpenAI-compatible HTTP server, one request at a time (a lock around `generate`). This is how most first deployments look.
2. **vLLM** — `vllm serve`, the OpenAI-compatible server with continuous batching and PagedAttention.
3. **Load client** — `asyncio` + streaming; fires 1 / 8 / 32 / 64 concurrent clients and measures **TTFT**, **output tokens/s**, **p95** and **failures** (60 s client timeout).

Before running: write down your prediction — *at 64 clients, how many times more tokens/s will vLLM deliver?*

Total run time ≈ 15–20 min, most of it installing vLLM and the naive server grinding through 64 clients.

## 0 · Setup

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
# vLLM goes into its OWN environment: it pins a newer torch (CUDA 13) than Colab's, and installing it
# into Colab's Python breaks the preinstalled torchaudio/transformers — the naive server needs those intact.
!pip -q install uv httpx fastapi uvicorn
!uv venv -q --python 3.12 /content/vllm-env && uv pip install -q --python /content/vllm-env vllm==0.29.0
!/content/vllm-env/bin/vllm --version

In [ ]:
import os, sys, json, time, asyncio, subprocess, statistics, signal
import httpx

MODEL = "Qwen/Qwen2.5-1.5B-Instruct"   # small, ungated, fits a T4 many times over
PORT_NAIVE, PORT_VLLM = 8001, 8000
CONCURRENCY = [1, 8, 32, 64]
MAX_TOKENS = 128          # output length per request
CLIENT_TIMEOUT = 60.0     # a client gives up after 60 s — like a real load balancer
PROMPTS = [
    "Explain in a few sentences why the sky is blue.",
    "Give three tips for writing readable Python code.",
    "What is a hash table? Answer briefly.",
    "Summarize the plot of a heist movie you would like to see.",
    "Why do GPUs help deep learning? Keep it short.",
    "Describe a good breakfast for a busy morning.",
    "What is the difference between a process and a thread?",
    "Write a short haiku about a busy kitchen.",
]
RESULTS = {}   # RESULTS[server][concurrency] = metrics dict
print("config ok")

## 1 · The naive server

A deliberately honest "first deployment": `pipeline` loaded once, a `threading.Lock` around generation (the pipeline is not safe to call concurrently, and one GPU stream at a time is what people get anyway), streaming via `TextIteratorStreamer`, OpenAI-style SSE chunks so the **same** client can talk to both servers.

In [ ]:
NAIVE_SERVER = r'''
import asyncio, json, queue, threading, time, torch, uvicorn
from fastapi import FastAPI, Request
from fastapi.responses import StreamingResponse
from transformers import pipeline, TextIteratorStreamer, StoppingCriteria, StoppingCriteriaList

MODEL = "%s"
pipe = pipeline("text-generation", model=MODEL, torch_dtype=torch.float16, device=0)
tok = pipe.tokenizer
lock = threading.Lock()
app = FastAPI()

@app.get("/v1/models")
def models():
    return {"data": [{"id": MODEL}]}

class Cancelled(StoppingCriteria):              # stop generating once the client has gone away
    def __init__(self, ev): self.ev = ev
    def __call__(self, *args, **kwargs): return self.ev.is_set()

@app.post("/v1/chat/completions")
async def chat(req: Request):
    body = await req.json()
    streamer = TextIteratorStreamer(tok, skip_prompt=True, skip_special_tokens=True)
    gone = threading.Event()
    def work():
        with lock:                                   # one order at a time
            if gone.is_set():                        # the guest left while waiting in the queue
                streamer.end(); return
            pipe(body["messages"], max_new_tokens=body.get("max_tokens", 128), do_sample=False,
                 streamer=streamer, stopping_criteria=StoppingCriteriaList([Cancelled(gone)]))
    threading.Thread(target=work, daemon=True).start()
    async def sse():
        # Poll the streamer's queue without blocking: no thread per waiting guest (a thread pool of
        # a few workers would starve the one stream that is actually cooking).
        try:
            while True:
                try:
                    piece = streamer.text_queue.get_nowait()
                except queue.Empty:
                    if await req.is_disconnected():  # the guest gave up: stop cooking their order
                        return
                    await asyncio.sleep(0.02)
                    continue
                if piece is streamer.stop_signal:
                    break
                if piece:
                    chunk = {"choices": [{"delta": {"content": piece}, "index": 0}]}
                    yield "data: " + json.dumps(chunk) + "\n\n"
            yield "data: [DONE]\n\n"
        finally:
            gone.set()                               # normal end, disconnect or cancellation
    return StreamingResponse(sse(), media_type="text/event-stream")

uvicorn.run(app, host="0.0.0.0", port=%d, log_level="warning")
''' % (MODEL, PORT_NAIVE)
open("naive_server.py", "w").write(NAIVE_SERVER)

In [ ]:
def start(cmd, log):
    return subprocess.Popen(cmd, stdout=open(log, "w"), stderr=subprocess.STDOUT, preexec_fn=os.setsid)

def gpu_used_mib():
    out = subprocess.run("nvidia-smi --query-gpu=memory.used --format=csv,noheader,nounits",
                         shell=True, capture_output=True, text=True).stdout
    return int(out.strip().splitlines()[0])

def stop(proc):
    for sig in (signal.SIGTERM, signal.SIGKILL):     # graceful first, then for sure
        if proc.poll() is not None:
            break
        try:
            os.killpg(os.getpgid(proc.pid), sig); proc.wait(timeout=30)
        except Exception:
            pass
    for _ in range(60):                              # the next server needs the VRAM back
        if gpu_used_mib() < 600:
            break
        time.sleep(1)
    print(f"stopped; GPU memory in use: {gpu_used_mib()} MiB")

def wait_ready(port, proc, log, timeout=900):
    t0 = time.time()
    while time.time() - t0 < timeout:
        if proc.poll() is not None:
            print(open(log).read()[-3000:]); raise RuntimeError("server died — log above")
        try:
            if httpx.get(f"http://localhost:{port}/v1/models", timeout=2).status_code == 200:
                print(f"ready on :{port} after {time.time()-t0:.0f} s"); return
        except Exception:
            pass
        time.sleep(3)
    raise TimeoutError(f"server on :{port} not ready")

naive = start([sys.executable, "naive_server.py"], "naive.log")
wait_ready(PORT_NAIVE, naive, "naive.log")

## 2 · The load client

Every client sends **one** streaming chat request at the same moment. Per request we record:

- **TTFT** — time until the first content chunk arrives; the primary p95 uses successful requests only, matching the judge;
- **tokens** — output tokens (re-tokenized, so both servers are counted the same way);
- **ok / failed** — failed = timeout or error.

Aggregate **throughput** = total output tokens of successful requests ÷ wall time of the whole wave. Failures are reported separately. We also show an explicitly labelled timeout-capped p95 as a diagnostic, never as the judge metric.

In [ ]:
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained(MODEL)

async def one_request(client, port, i):
    body = {"model": MODEL, "max_tokens": MAX_TOKENS, "temperature": 0, "stream": True,
            "messages": [{"role": "user", "content": PROMPTS[i % len(PROMPTS)]}]}
    t0 = time.perf_counter(); ttft = None; text = []
    try:
        async with asyncio.timeout(CLIENT_TIMEOUT):
            async with client.stream("POST", f"http://localhost:{port}/v1/chat/completions", json=body) as r:
                r.raise_for_status()
                async for line in r.aiter_lines():
                    if not line.startswith("data: ") or line.endswith("[DONE]"):
                        continue
                    delta = json.loads(line[6:])["choices"][0]["delta"].get("content") or ""
                    if delta and ttft is None:
                        ttft = time.perf_counter() - t0
                    text.append(delta)
        joined = "".join(text)
        if ttft is None:
            return {"ok": False, "ttft": None, "tokens": 0,
                    "latency": time.perf_counter() - t0, "err": "EmptyResponse"}
        return {"ok": True, "ttft": ttft, "tokens": len(tok.encode(joined)),
                "latency": time.perf_counter() - t0}
    except Exception as e:
        return {"ok": False, "ttft": None, "tokens": 0, "latency": time.perf_counter() - t0, "err": type(e).__name__}

def p95(xs):
    xs = sorted(xs); return xs[min(len(xs) - 1, int(round(0.95 * (len(xs) - 1))))] if xs else float("nan")

async def wave(port, n):
    limits = httpx.Limits(max_connections=n + 8)
    async with httpx.AsyncClient(timeout=None, limits=limits) as client:
        t0 = time.perf_counter()
        res = await asyncio.gather(*[one_request(client, port, i) for i in range(n)])
        wall = time.perf_counter() - t0
    ok = [r for r in res if r["ok"]]
    ttfts_success = [r["ttft"] for r in ok if r["ttft"] is not None]
    ttfts_capped = [r["ttft"] if r["ok"] and r["ttft"] is not None else CLIENT_TIMEOUT for r in res]
    return {"clients": n, "ok": len(ok), "failed": n - len(ok), "wall_s": wall,
            "tok_per_s": sum(r["tokens"] for r in ok) / wall,
            "ttft_p50_success": statistics.median(ttfts_success) if ttfts_success else float("nan"),
            "ttft_p95_success": p95(ttfts_success),
            "ttft_p95_capped": p95(ttfts_capped)}

async def run_all(name, port, proc):
    assert proc.poll() is None, f"{name} server is not running — look at its log above before measuring"
    RESULTS[name] = {}
    await wave(port, 1)                       # warm-up, not recorded
    for n in CONCURRENCY:
        m = await wave(port, n); RESULTS[name][n] = m
        print(f"{name:8s} c={n:3d}  {m['tok_per_s']:8.1f} tok/s   successful TTFT p50 {m['ttft_p50_success']:6.2f} s  "
              f"p95 {m['ttft_p95_success']:6.2f} s   ok {m['ok']}/{n}   wall {m['wall_s']:.0f} s")

## 3 · Run the naive server

Watch the p95 TTFT column as concurrency grows, and the `ok` column at 64. Every client waits for all the orders in front of it.

In [ ]:
await run_all("pipeline", PORT_NAIVE, naive)
stop(naive)      # free the GPU for vLLM

**Read it before moving on.** Does throughput grow with the number of clients? It should stay roughly flat: the server does the same amount of work per second no matter how many people are waiting, so extra clients only add waiting time. The p95 TTFT grows roughly linearly with the number of clients. At 64 clients some requests probably hit the 60 s timeout, and those guests simply left.

## 4 · The same model on vLLM

`--dtype half` because the T4 (Turing) has no bfloat16. `--max-model-len 2048` keeps the KV-cache reservation modest.

In [ ]:
vllm = start(["/content/vllm-env/bin/vllm", "serve", MODEL, "--port", str(PORT_VLLM), "--dtype", "half",
              "--max-model-len", "2048", "--gpu-memory-utilization", "0.85"], "vllm.log")
wait_ready(PORT_VLLM, vllm, "vllm.log")

After the recorded benchmark waves, we run one dedicated 64-client verification wave while polling vLLM's Prometheus endpoint `/metrics`. We record `vllm:num_requests_running`, the number of requests in the current batch. This extra wave is not included in the table; it is a direct mechanism check.

In [ ]:
import re

async def poll_running(stop_evt, samples):
    async with httpx.AsyncClient() as c:
        while not stop_evt.is_set():
            try:
                txt = (await c.get(f"http://localhost:{PORT_VLLM}/metrics", timeout=2)).text
                m = re.search(r'^vllm:num_requests_running\{[^}]*\}\s+([0-9.]+)', txt, re.M)
                if m: samples.append(float(m.group(1)))
            except Exception:
                pass
            await asyncio.sleep(0.25)

await run_all("vLLM", PORT_VLLM, vllm)

samples, evt = [], asyncio.Event()
poller = asyncio.create_task(poll_running(evt, samples))
await wave(PORT_VLLM, 64)
evt.set(); await poller
print("max requests running in one batch during the 64-client wave:", max(samples) if samples else "n/a")

## 5 · The table for the slide

In [ ]:
print("p95 TTFT below is computed over successful requests; failures are a separate column")
print(f"{'clients':>7} | {'pipeline tok/s':>14} | {'vLLM tok/s':>10} | {'speed-up':>8} | "
      f"{'pipeline p95 TTFT':>17} | {'vLLM p95 TTFT':>13} | {'pipeline failed':>15} | {'vLLM failed':>11}")
for n in CONCURRENCY:
    a, b = RESULTS["pipeline"][n], RESULTS["vLLM"][n]
    print(f"{n:>7} | {a['tok_per_s']:>14.1f} | {b['tok_per_s']:>10.1f} | {(format(b['tok_per_s'] / a['tok_per_s'], '7.1f') + 'x') if a['tok_per_s'] else '    n/a '} | "
          f"{a['ttft_p95_success']:>15.2f} s | {b['ttft_p95_success']:>11.2f} s | {a['failed']:>15} | {b['failed']:>11}")
json.dump(RESULTS, open("lecture0_demo_results.json", "w"), indent=1)

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
for name, style in [("pipeline", "o--"), ("vLLM", "o-")]:
    ax[0].plot(CONCURRENCY, [RESULTS[name][n]["tok_per_s"] for n in CONCURRENCY], style, label=name)
    ax[1].plot(CONCURRENCY, [RESULTS[name][n]["ttft_p95_success"] for n in CONCURRENCY], style, label=name)
ax[0].set(title="Throughput — plates per hour", xlabel="concurrent clients", ylabel="output tokens / s")
ax[1].set(title="p95 TTFT — successful requests", xlabel="concurrent clients", ylabel="seconds")
for a in ax: a.set_xscale("log", base=2); a.set_xticks(CONCURRENCY, CONCURRENCY); a.grid(alpha=.3); a.legend()
plt.tight_layout(); plt.show()

**What to read off the plots**

- **Throughput.** The pipeline's line stays roughly flat. vLLM's line keeps rising with concurrency: one decode step reads the weights once and produces a token for every request in the batch (slide 11).
- **p95 TTFT.** This curve uses successful requests only, so always read it together with the failure count. The pipeline's p95 grows roughly linearly while requests still complete. vLLM's stays low because new requests join the running batch at the next step instead of waiting for the whole queue.
- **Compare with your prediction.** Did you over- or underestimate the gap? The T4 has about 320 GB/s of memory bandwidth. The judge's RTX 5070 Ti has about 896 GB/s, so absolute numbers will be larger there, but the shape of both curves stays the same.

Topics 5–7 explain *why* these curves look this way and how to move them.

In [ ]:
stop(vllm)